In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Funzione per calcolare la media mobile pesata
def media_mobile_pesata(df, window):
    df['media_mobile_pesata'] = df['scarto_percentuale'].rolling(window=window, min_periods=1).mean()
    return df

# Funzione per calcolare la media pesata complessiva
def calcola_media_pesata_complessiva(df):
    return df['scarto_percentuale'].mean()

# Funzione per calcolare la media pesata stagionale
def calcola_media_stagionale(df):
    # Aggiungiamo una colonna per la stagione (es: inverno, primavera, estate, autunno)
    df['stagione'] = pd.to_datetime(df['data']).dt.month % 12 // 3 + 1
    stagioni = {1: 'Inverno', 2: 'Primavera', 3: 'Estate', 4: 'Autunno'}
    media_stagionale = df.groupby('stagione')['scarto_percentuale'].mean().rename(index=stagioni).to_dict()
    return media_stagionale

# Funzione per calcolare il DataFrame del "scarto percentuale"
def calcola_media_scarto(file_csv):
    df = pd.read_csv(file_csv)
    df['data'] = pd.to_datetime(df['data'])
    return df

# Funzione per contare le date per ogni stagione in ogni anno
def conta_date_per_stagione(df):
    df['anno'] = df['data'].dt.year
    df['stagione'] = pd.to_datetime(df['data']).dt.month % 12 // 3 + 1
    conteggio = df.groupby(['anno', 'stagione']).size().unstack(fill_value=0)
    print(conteggio)

# Definizione del percorso del font
font_path = r'C:\Users\vitto\Desktop\ADLaM_Display.ttf'  # Assicurati che il percorso sia corretto e che includa l'estensione del file

# Funzione principale per la visualizzazione del grafico
def plot_scarto_percentuale(media_scarto_df, window):
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()

    # Calcola la media mobile pesata a 'window' giorni
    media_scarto_df = media_mobile_pesata(media_scarto_df, window)

    # Calcola la media pesata complessiva
    media_pesata_complessiva = calcola_media_pesata_complessiva(media_scarto_df)

    # Calcola la media pesata stagionale
    media_stagionale = calcola_media_stagionale(media_scarto_df)

    # Crea il grafico
    plt.figure(figsize=(10,6))

    # Plot scarto percentuale giornaliero
    plt.plot(media_scarto_df['data'], media_scarto_df['scarto_percentuale'], marker='o', linestyle='None', color='#00ff00', label='Scarto Percentuale')

    # Plot media mobile pesata a 'window' giorni
    plt.plot(media_scarto_df['data'], media_scarto_df['media_mobile_pesata'], color='#2e8b57', linestyle='-', linewidth=2, label=f'Media Mobile Pesata {window} Giorni')

    # Aggiunge titolo e etichette agli assi
    plt.xlabel('DATA', fontsize=16)
    plt.ylabel('Media Scarto Percentuale (%)', fontsize=16)
    plt.ylim(0, 20)
    
    # Imposta la dimensione e il font delle tacche degli assi
    plt.xticks(rotation=45, fontsize=12)
    plt.yticks(fontsize=12)

    # Aggiunge la griglia e la legenda
    plt.grid(True)
    plt.legend()

    # Migliora la gestione degli spazi
    plt.tight_layout()
    plt.savefig('scarto_percentuale_stazioni_ERA5.png', bbox_inches='tight')

    # Mostra il grafico
    plt.show()

    # Stampa la media pesata complessiva e le medie stagionali
    print(f"Media pesata complessiva: {media_pesata_complessiva:.2f}%")
    for stagione, media in media_stagionale.items():
        print(f"Media pesata {stagione}: {media:.2f}%")

# Esecuzione del codice
file_csv = 'ERA_PV_stazioni_clearsky.csv'  # Inserisci qui il percorso del tuo file CSV
media_scarto_df = calcola_media_scarto(file_csv)

# Visualizza il grafico con la media mobile pesata e la media pesata complessiva
plot_scarto_percentuale(media_scarto_df, window=20)

# Conta le date per ogni stagione in ogni anno
conta_date_per_stagione(media_scarto_df)   